<a href="https://colab.research.google.com/github/jeremy26/hydranets_course/blob/main/Module_3_Advanced_Heads_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3: Adding New Heads to a HydraNet — Lab

In this module, you'll learn how to **add new task heads** to an existing, pre-trained HydraNet. This is the real power of multi-task learning: once you have a strong shared backbone, adding a new task is fast and cheap.

**What you'll learn:**
1. How to freeze a pre-trained backbone and add new heads
2. How to design a head for a specific task
3. How to train only the new head using pre-computed features (~5-15 min on Colab)

**Demo:** The instructor shows how to build a **Lane Detection** head

**Your Lab:** Build your own head! Choose from:
- 2D Object Detection (CenterNet-style)
- Trajectory / Steering Prediction
- Drivable Area Segmentation
- Or your own idea!

**Architecture reminder:**
```
Image -> [FROZEN Backbone] -> [FROZEN Context] -> [FROZEN Neck] -> [YOUR NEW HEAD] -> Prediction
```

# 1 — Setup

In [ ]:
!git clone https://github.com/jeremy26/hydranets_course.git 2>/dev/null || true
%cd hydranets_course
!pip install -q torchvision pillow matplotlib numpy tqdm

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
import torch.nn as nn
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Download data and pre-computed features from Module 2
!wget -q https://hydranets-data.s3.eu-west-3.amazonaws.com/bdd100k_hydranet_subset.zip \
    && unzip -q bdd100k_hydranet_subset.zip -d data \
    && rm bdd100k_hydranet_subset.zip

!wget -q https://hydranets-data.s3.eu-west-3.amazonaws.com/precomputed_module2.zip \
    && unzip -q precomputed_module2.zip -d precomputed \
    && rm precomputed_module2.zip

DATA_ROOT = "data/bdd100k"

# 2 — Load Pre-Computed Features

Instead of running the full backbone on every training iteration, we load **pre-computed neck features** saved from Module 2. This means:
- No GPU needed for the backbone
- Training a new head takes **minutes, not hours**
- Students can iterate quickly on head designs

In [ ]:
# Load pre-computed features from Module 2
data = torch.load('precomputed/train_features.pt', map_location='cpu')

neck_features = data['neck']          # (N, 256, H/4, W/4)
skip_features = data['features_0']    # (N, 32, H/2, W/2)
filenames = data['filenames']

print(f"Loaded {len(filenames)} pre-computed samples")
print(f"Neck features shape:     {list(neck_features.shape)}")
print(f"Skip features shape:     {list(skip_features.shape)}")
print(f"\nThese are the outputs of the FROZEN backbone + context + neck.")
print(f"Your new head will take these as input.")

In [ ]:
from torch.utils.data import Dataset, DataLoader

class PrecomputedDataset(Dataset):
    """Dataset that loads pre-computed backbone features + task labels.
    No backbone forward pass needed — just load tensors!"""

    def __init__(self, neck_features, skip_features, filenames,
                 label_dir, label_suffix='.png', label_type='mask',
                 target_size=None):
        self.neck = neck_features
        self.skip = skip_features
        self.filenames = filenames
        self.label_dir = label_dir
        self.label_suffix = label_suffix
        self.label_type = label_type
        self.target_size = target_size

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        neck = self.neck[idx]
        skip = self.skip[idx]

        label_path = os.path.join(self.label_dir, self.filenames[idx] + self.label_suffix)

        if self.label_type == 'mask' and os.path.exists(label_path):
            from PIL import Image
            label = Image.open(label_path)
            if self.target_size:
                label = label.resize(
                    (self.target_size[1], self.target_size[0]),
                    Image.NEAREST
                )
            label = torch.from_numpy(np.array(label)).long()
        else:
            h, w = neck.shape[1], neck.shape[2]
            if self.target_size:
                h, w = self.target_size
            label = torch.zeros((h, w), dtype=torch.long)

        return {'neck': neck, 'skip': skip, 'label': label}

print("PrecomputedDataset ready!")

# 3 — Demo: Building a Lane Detection Head

The instructor demonstrates how to add a **Lane Detection** head to the HydraNet. This is the pattern you'll follow in the lab.

## Step 1: Design the Head

The `LanesHead` is one of the simplest heads — it operates at **neck resolution** (H/4 x W/4) since lanes are thin structures that don't need full resolution.

```
Neck (256ch, H/4, W/4) -> Conv 3x3 -> GELU -> Conv 3x3 -> GELU -> Conv 3x3 -> Output (3ch, H/4, W/4)
```

3 output channels: background, left lane, right lane.

In [ ]:
from models.heads import LanesHead

lanes_head = LanesHead(num_classes=3).to(device)

# Count parameters — heads are tiny!
n_params = sum(p.numel() for p in lanes_head.parameters())
print(f"LanesHead parameters: {n_params:,}")
print(f"That's tiny compared to the backbone — this is why training is fast!")

# Test forward pass
dummy_neck = torch.randn(2, 256, 80, 160).to(device)  # H/4 x W/4 for 320x640 input
out = lanes_head(dummy_neck)
print(f"\nInput (neck):  {list(dummy_neck.shape)}")
print(f"Output (lanes): {list(out.shape)}")

## Step 2: Create DataLoader with Pre-computed Features

In [ ]:
# Create dataset using pre-computed features
lane_label_dir = os.path.join(DATA_ROOT, 'labels', 'lane', 'masks', 'train')

# Lanes operate at neck resolution (H/4 x W/4)
neck_h, neck_w = neck_features.shape[2], neck_features.shape[3]

lane_dataset = PrecomputedDataset(
    neck_features=neck_features,
    skip_features=skip_features,
    filenames=filenames,
    label_dir=lane_label_dir,
    label_suffix='.png',
    target_size=(neck_h, neck_w),  # Lanes at neck resolution
)

# Split 80/20 for train/val
n_train = int(0.8 * len(lane_dataset))
n_val = len(lane_dataset) - n_train
train_set, val_set = torch.utils.data.random_split(lane_dataset, [n_train, n_val])

train_loader = DataLoader(train_set, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=16, shuffle=False, num_workers=0)

print(f"Train: {len(train_set)} | Val: {len(val_set)}")
print(f"Label resolution: {neck_h} x {neck_w} (neck resolution = H/4 x W/4)")

## Step 3: Train the Head

Notice: we're only training the head — no backbone, no context, no neck. This is why it's fast!

In [ ]:
# Training setup
lanes_head = LanesHead(num_classes=3).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = torch.optim.Adam(lanes_head.parameters(), lr=1e-3)

NUM_EPOCHS = 20

def train_head(head, train_loader, val_loader, criterion, optimizer, device, num_epochs=20):
    """Generic training loop for any head using pre-computed features."""
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(num_epochs):
        # Train
        head.train()
        train_loss = 0
        for batch in train_loader:
            neck = batch['neck'].to(device)
            label = batch['label'].to(device)

            pred = head(neck)
            loss = criterion(pred, label)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validate
        head.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                neck = batch['neck'].to(device)
                label = batch['label'].to(device)
                pred = head(neck)
                val_loss += criterion(pred, label).item()
        val_loss /= len(val_loader)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{num_epochs} | "
                  f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    return history

print("Training lane detection head...")
history = train_head(lanes_head, train_loader, val_loader, criterion, optimizer, device, NUM_EPOCHS)
print("Done!")

In [ ]:
# Plot training curves
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(history['train_loss'], label='Train')
ax.plot(history['val_loss'], label='Val')
ax.set_title('Lane Detection Head — Training Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout()
plt.show()

## Step 4: Visualize Lane Predictions

In [ ]:
# Visualize predictions
LANE_COLORS = np.array([
    [0, 0, 0],       # background
    [255, 0, 0],     # left lane
    [0, 0, 255],     # right lane
], dtype=np.uint8)

lanes_head.eval()
val_batch = next(iter(val_loader))

with torch.no_grad():
    pred = lanes_head(val_batch['neck'].to(device))
    pred = pred.argmax(dim=1).cpu().numpy()

fig, axes = plt.subplots(4, 3, figsize=(15, 16))
for i in range(4):
    # Load original image for reference
    from PIL import Image as PILImage
    from utils.data import INPUT_SIZE
    img_path = os.path.join(DATA_ROOT, 'images', '10k', 'train',
                            filenames[val_set.indices[i]] + '.jpg')
    if not os.path.exists(img_path):
        img_path = os.path.join(DATA_ROOT, 'images', 'train',
                                filenames[val_set.indices[i]] + '.jpg')

    if os.path.exists(img_path):
        img = PILImage.open(img_path).resize((INPUT_SIZE[1], INPUT_SIZE[0]))
        axes[i, 0].imshow(img)
    axes[i, 0].set_title('Input Image')
    axes[i, 0].axis('off')

    # Ground truth
    gt = val_batch['label'][i].numpy()
    gt_rgb = np.zeros((*gt.shape, 3), dtype=np.uint8)
    for c in range(len(LANE_COLORS)):
        gt_rgb[gt == c] = LANE_COLORS[c]
    axes[i, 1].imshow(gt_rgb)
    axes[i, 1].set_title('GT Lanes')
    axes[i, 1].axis('off')

    # Prediction
    pred_rgb = np.zeros((*pred[i].shape, 3), dtype=np.uint8)
    for c in range(len(LANE_COLORS)):
        pred_rgb[pred[i] == c] = LANE_COLORS[c]
    axes[i, 2].imshow(pred_rgb)
    axes[i, 2].set_title('Predicted Lanes')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

# 4 — Plug it into the Full HydraNet

Now let's see how the lane head fits into the complete HydraNet with all heads running together.
The `HydraNetExtended` class takes a pre-trained HydraNet and lets you add new heads on top.

In [ ]:
from models.hydranet import HydraNet, HydraNetExtended
from models.heads import LanesHead

# Load the pre-trained Module 2 HydraNet
base_model = HydraNet(num_seg_classes=19)
base_model.load_state_dict(torch.load('precomputed/hydranet_module2.pth', map_location=device))

# Create extended model with frozen backbone + our new lanes head
extended_model = HydraNetExtended(
    pretrained_hydranet=base_model,
    new_heads={'lanes': lanes_head}
)
extended_model = extended_model.to(device)

# Check what's trainable vs frozen
frozen = sum(p.numel() for p in extended_model.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in extended_model.parameters() if p.requires_grad)
print(f'Frozen parameters:    {frozen:,}')
print(f'Trainable parameters: {trainable:,}')
print(f'Ratio: only {100*trainable/(frozen+trainable):.1f}% of the network is trainable!')

# Test forward pass with a real image
dummy = torch.randn(1, 3, 320, 640).to(device)
outputs = extended_model(dummy)
print(f'\nOutputs: {list(outputs.keys())}')
for name, out in outputs.items():
    if isinstance(out, torch.Tensor):
        print(f'  {name}: {list(out.shape)}')
    elif isinstance(out, tuple):
        print(f'  {name}: {[list(o.shape) for o in out]}')

# 5 — YOUR TURN: Build Your Own Head!

Now it's your turn. Pick one of the tasks below and build a head for it.

**Remember the recipe:**
1. Design the head (a few conv layers)
2. Create a DataLoader with pre-computed features + your labels
3. Train (should take ~5-15 min)
4. Visualize results

**Your input is always:** `neck` = (B, 256, 80, 160) from the frozen backbone.

Choose your challenge:

## Option A: 2D Object Detection Head (CenterNet-style)

Predict object center heatmaps + bounding box offsets.

**Output:**
- Heatmap: (B, 10, H/4, W/4) — 10 BDD100K classes (car, truck, bus, person, etc.)
- Regression: (B, 4, H/4, W/4) — bbox offsets (dx, dy, w, h)

**Loss:** Focal Loss for heatmap + L1 for regression

**Hint:** Look at `DetectionHead` in `models/heads.py` for the architecture.

In [ ]:
# ============================================================
# OPTION A: 2D Object Detection Head
# ============================================================
# Uncomment and complete this section if you choose Option A

# from models.heads import DetectionHead
#
# class FocalLoss(nn.Module):
#     """Focal loss for heatmap prediction (reduces easy negative impact)."""
#     def __init__(self, alpha=2, beta=4):
#         super().__init__()
#         self.alpha = alpha
#         self.beta = beta
#
#     def forward(self, pred, target):
#         pos_mask = target.eq(1).float()
#         neg_mask = target.lt(1).float()
#
#         pos_loss = -torch.log(pred + 1e-8) * torch.pow(1 - pred, self.alpha) * pos_mask
#         neg_loss = -torch.log(1 - pred + 1e-8) * torch.pow(pred, self.alpha) * \
#                    torch.pow(1 - target, self.beta) * neg_mask
#
#         num_pos = pos_mask.sum().clamp(min=1)
#         return (pos_loss.sum() + neg_loss.sum()) / num_pos
#
# det_head = DetectionHead(num_classes=10).to(device)
# focal_loss = FocalLoss()
# reg_loss = nn.L1Loss()
# optimizer = torch.optim.Adam(det_head.parameters(), lr=1e-3)
#
# # TODO: Create a PrecomputedDataset for detection labels
# # TODO: Write the training loop (hint: heatmap, regression = det_head(neck))
# # TODO: Visualize predicted heatmaps overlaid on images

## Option B: Trajectory / Steering Prediction Head

Predict the steering angle as a classification over 61 discrete bins (-30° to +30°).
This is exactly how [Autoware Vision Pilot](https://github.com/autowarefoundation/autoware_vision_pilot) does it.

**Output:** (B, 61) — logits over steering angle bins

**Loss:** Cross-Entropy (classification)

**Hint:** Look at `TrajectoryHead` in `models/heads.py` — it uses AdaptiveAvgPool + FC layers.

In [ ]:
# ============================================================
# OPTION B: Trajectory / Steering Prediction Head
# ============================================================
# Uncomment and complete this section if you choose Option B

# from models.heads import TrajectoryHead
#
# traj_head = TrajectoryHead(num_bins=61).to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(traj_head.parameters(), lr=1e-3)
#
# # BDD100K GPS/IMU data gives us steering angles
# # Labels: integer class index in [0, 60] mapping to [-30, +30] degrees
#
# # TODO: Load steering angle labels from BDD100K driving logs
# # TODO: Create a simple dataset that pairs neck features with steering labels
# # TODO: Train the head
# # TODO: Plot predicted vs actual steering angle distribution

## Option C: Drivable Area Segmentation

Predict which pixels are drivable (road you can drive on).

**Output:** (B, 3, H/4, W/4) — 3 classes: not drivable, direct (your lane), alternative (other lanes)

**Loss:** Cross-Entropy

**Hint:** This is structurally identical to `LanesHead` — just change the labels!

In [ ]:
# ============================================================
# OPTION C: Drivable Area Segmentation Head
# ============================================================
# Uncomment and complete this section if you choose Option C

# class DrivableAreaHead(nn.Module):
#     """Drivable area segmentation — same pattern as LanesHead."""
#     def __init__(self, num_classes=3):
#         super().__init__()
#         self.GeLU = nn.GELU()
#         self.decode_layer_0 = nn.Conv2d(256, 256, 3, 1, 1)
#         self.decode_layer_1 = nn.Conv2d(256, 128, 3, 1, 1)
#         self.output_layer = nn.Conv2d(128, num_classes, 3, 1, 1)
#
#     def forward(self, neck, features=None):
#         d0 = self.GeLU(self.decode_layer_0(neck))
#         d1 = self.GeLU(self.decode_layer_1(d0))
#         return self.output_layer(d1)
#
# drivable_head = DrivableAreaHead(num_classes=3).to(device)
# criterion = nn.CrossEntropyLoss(ignore_index=255)
# optimizer = torch.optim.Adam(drivable_head.parameters(), lr=1e-3)
#
# # TODO: Create PrecomputedDataset pointing to drivable area masks
# # TODO: Train using the train_head() function from the demo
# # TODO: Visualize: green = direct, blue = alternative, black = not drivable

## Option D: Build Your Own Head!

Design a head for any task you can think of. Some ideas:
- **Weather classification** (sunny, rainy, foggy, etc.) — BDD100K has these labels
- **Time of day** (daytime, nighttime, dawn/dusk) — BDD100K has these labels
- **Free-space estimation** — binary mask of where the car can drive
- **Traffic sign detection** — find and classify traffic signs

**Template below:**

In [ ]:
# ============================================================
# OPTION D: Your Own Head
# ============================================================

# class MyHead(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.GeLU = nn.GELU()
#         # YOUR LAYERS HERE
#         # Input: neck (B, 256, 80, 160)
#         # Output: whatever your task needs!
#
#     def forward(self, neck, features=None):
#         # YOUR FORWARD PASS HERE
#         pass
#
# my_head = MyHead().to(device)
# criterion = ...  # YOUR LOSS
# optimizer = torch.optim.Adam(my_head.parameters(), lr=1e-3)
#
# # Train it!
# history = train_head(my_head, train_loader, val_loader,
#                      criterion, optimizer, device, num_epochs=20)

# 6 — Combine All Heads into the Final HydraNet

Once you've trained your head, plug it into the full model alongside the pre-trained ones!

In [ ]:
# Combine everything into one HydraNet
from models.hydranet import HydraNet, HydraNetExtended

# Load pre-trained base
base_model = HydraNet(num_seg_classes=19)
base_model.load_state_dict(torch.load('precomputed/hydranet_module2.pth', map_location=device))

# Build extended model with ALL heads
extended = HydraNetExtended(
    pretrained_hydranet=base_model,
    new_heads={
        'lanes': lanes_head,  # From demo
        # 'detection': det_head,     # Uncomment if you built Option A
        # 'trajectory': traj_head,    # Uncomment if you built Option B
        # 'drivable': drivable_head,  # Uncomment if you built Option C
        # 'custom': my_head,          # Uncomment if you built Option D
    }
)
extended = extended.to(device)

# Run inference
dummy = torch.randn(1, 3, 320, 640).to(device)
outputs = extended(dummy)

print('Full HydraNet outputs:')
for name, out in outputs.items():
    if isinstance(out, torch.Tensor):
        print(f'  {name}: {list(out.shape)}')
    elif isinstance(out, tuple):
        print(f'  {name}: {[list(o.shape) for o in out]}')

# 7 — FPS: How Fast is Our Multi-Head HydraNet?

In [ ]:
import time

extended.eval()
dummy = torch.randn(1, 3, 320, 640).to(device)

# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = extended(dummy)

if torch.cuda.is_available():
    torch.cuda.synchronize()

start = time.time()
n_runs = 100
for _ in range(n_runs):
    with torch.no_grad():
        _ = extended(dummy)

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.time() - start
fps = n_runs / elapsed
n_heads = len(extended.heads)
print(f'HydraNet with {n_heads} head(s): {fps:.1f} FPS ({1000*elapsed/n_runs:.1f} ms/frame)')
print(f'All {n_heads} task outputs computed in a SINGLE forward pass!')
print(f'\nCompare this to running {n_heads} separate models — HydraNet is ~{n_heads}x more efficient.')

# Summary

In this module, you learned how to:

1. **Freeze a pre-trained backbone** — use `FrozenBackbone` to lock down trained weights
2. **Design lightweight heads** — a few conv layers is all you need
3. **Train heads in minutes** — using pre-computed features, no backbone needed
4. **Combine heads into a HydraNet** — `HydraNetExtended` manages multiple heads

**Key takeaway:** The power of multi-task learning is that the shared backbone does the heavy lifting.
Adding a new task is cheap — just design a head and train it!

**Architecture (Autoware Vision Pilot pattern):**
```
                                    ┌─> Segmentation (Module 2)
                                    ├─> Depth (Module 2)
Image -> Backbone -> Context -> Neck ├─> Lanes (Module 3 demo)
                                    ├─> Detection (your lab)
                                    └─> Trajectory (your lab)
```

This is exactly how modern autonomous driving perception stacks work in production.